# ME324 · Lab 9 — Text-gen 2/4: recurrent networks

**Lecture 9 · "Building your own text-generating AI (2/4)" · 2026-08-13**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-09-rnn.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

---

> **Turn the GPU on first.** In Colab: **Runtime → Change runtime type → Hardware
> accelerator → GPU**, then run the cells in order, top to bottom.

### Where we are

In **Lab 8** you built a complete text pipeline on *tiny-shakespeare* and trained a
**bigram** model — yielding gibberish samples. Today we'll keep the whole
pipeline but swap in a model that has memory: a **recurrent neural network (RNN)**. 

### Goals for today

1. **Hand-roll** the vanilla RNN recurrence $h_t = \tanh(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b_h)$ and loop it over a sequence.
2. See, numerically, *why* the **gradient vanishes** over many steps.
3. Wrap the cell as an `nn.Module`
4. Train the gated **`nn.GRU`** with the unchanged Lab-8 loop and compare it to the bigram.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** Sections 2–3 — the recurrence, the hand-rolled model, and the GRU.
- **Stretch / take-home — skip if short on time:** Section 4's training runs and comparisons — start them in the room, read the samples at home.

_There are **eight** `# TODO` cells this time — two of them whole functions with a spec in the comments. Worked answers are in the **Solutions** section at the bottom._

## Run me first — setup

Imports, the course seed (1337), and the GPU check. If it prints `device: cpu`, go to
**Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU** and re-run.

In [ ]:
import math, time
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)          # reproducibility (same seed as every ME324 lab)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime > Change runtime type > GPU (the lab still runs on CPU, just slower).')

## Section 1 · The Lab-8 pipeline (reused verbatim)

Everything here is copied unchanged from Lab 8: the tiny-shakespeare download,
`build_char_pipeline`, the generic `train_model` loop, and the **bigram** — our baseline
to beat.

In [ ]:
# The SAME corpus as Lab 8: ~1.1M characters of Shakespeare, character-level.
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('total characters:', len(text))
print('---- first 250 characters ----')
print(text[:250])

### The pipeline function (verbatim from Lab 8)

One dictionary bundling the tokenizer (`encode`, `decode`), the train/val tensors,
`get_batch(split)`, and `estimate_loss(model)`.

In [ ]:
def build_char_pipeline(text, block_size=8, batch_size=32, device="cpu", seed=1337):
    """Char-level tokenizer + batcher + loss estimator. (Lab 8, unchanged.)"""
    torch.manual_seed(seed)
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}        # char  -> integer id
    itos = {i: ch for i, ch in enumerate(chars)}        # id    -> char
    encode = lambda s: [stoi[c] for c in s]             # string -> list[int]
    decode = lambda l: ''.join(itos[i] for i in l)      # list[int] -> string
    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]           # 90% train / 10% val

    def get_batch(split):
        d = train_data if split == 'train' else val_data
        ix = torch.randint(len(d) - block_size, (batch_size,))      # random starts
        x = torch.stack([d[i:i + block_size] for i in ix])          # (B, block_size)
        y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # shifted by one
        return x.to(device), y.to(device)

    @torch.no_grad()
    def estimate_loss(model, eval_iters=200):
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch(split)
                _, loss = model(xb, yb)              
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    return dict(vocab_size=vocab_size, stoi=stoi, itos=itos, encode=encode,
                decode=decode, get_batch=get_batch, estimate_loss=estimate_loss,
                block_size=block_size, device=device)

### The training loop (verbatim from Lab 8)

`train_model` never mentions bigrams or RNNs — it only needs a model that returns
`(logits, loss)`.

In [ ]:
def train_model(model, get_batch, estimate_loss, max_iters=3000, eval_interval=500,
                lr=1e-3, device="cpu"):
    """Generic AdamW training loop. Reused by every LM in Labs 8-11."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model)
            print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)                 # forward: model -> (logits, loss)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()                         # backward: autograd
        optimizer.step()                        # update weights
    return model

### Build the pipeline

Block size goes up from 8 to 32: each training example is a 32-character window, and the
target is the same window shifted one character right — a next-character prediction at
*every* position.

**Before you run the next cell: what shapes will `xb` and `yb` have?**

In [ ]:
# TODO: before you run this cell, predict the exact shapes of xb and yb.
# Write your prediction in the comment at the bottom, then run and check.
block_size = 32          # how many characters of context per training example
batch_size = 64          # how many windows per batch

P = build_char_pipeline(text, block_size=block_size, batch_size=batch_size, device=device)
print('vocab_size:', P['vocab_size'], '(distinct characters)')

xb, yb = P['get_batch']('train')
print('inputs  xb:', tuple(xb.shape))
print('targets yb:', tuple(yb.shape))
print('decoded first input :', repr(P['decode'](xb[0].tolist())))
print('decoded first target:', repr(P['decode'](yb[0].tolist())))

# Your prediction — xb: (?, ?)   yb: (?, ?)   Why are they the same shape?
#

### The Lab-8 bigram model (our baseline)

Exactly as you built it: `Embedding(vocab, vocab)` maps each character id straight to a
row of next-character scores — no memory of anything earlier. We'll train it in Section 4.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # one row of next-char scores per character: no context, no memory.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)            # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]                       # last position only
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print('Bigram defined — same model as Lab 8.')

## Section 2 · Hand-rolling a recurrent cell

An **RNN** reads the sequence one character at a time while carrying a **hidden state**
$h_t$ — a running summary of everything seen so far. From the lecture:

$$
h_t = \tanh\!\big(\,W_{xh}\,x_t + W_{hh}\,h_{t-1} + b_h\,\big)
$$

$W_{xh}$ mixes in the new input $x_t$; $W_{hh}$ mixes in the old memory $h_{t-1}$;
$\tanh$ squashes the result into $(-1, 1)$; and **the same weights are used at every
step**. Because $h_t$ depends on $h_{t-1}$, and $h_{t-1}$ on $h_{t-2}$, the chain runs
back to the start of the sequence (i.e. it has memory).

### A one-dimensional RNN, by hand

The lecture's plenary: hidden dimension 1, $W_{xh}=1.0$, $W_{hh}=0.5$, $b_h=0$, $h_0=0$,
inputs $x = [1.0,\,-0.5,\,2.0]$. The one-line body of `rnn_step` is yours; done right,
you get $h_1 \approx 0.762$, $h_2 \approx -0.119$, $h_3 \approx 0.959$ — the numbers from
the board.

In [ ]:
def rnn_step(x_t, h_prev, W_xh=1.0, W_hh=0.5, b_h=0.0):
    # TODO: one scalar RNN step, h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h).
    #       math.tanh is your friend here — everything is a plain Python float.
    return ____                   # <-- replace ____

h = 0.0                                  # h_0 = 0
for x_t in [1.0, -0.5, 2.0]:
    h = rnn_step(x_t, h)
    print(f'x_t = {x_t:+.1f}   ->   h = {h:+.3f}')

### The hidden state really is a memory

Two sequences ending with the same input ($x=1.0$) but different histories. The bigram
would give identical results; the RNN's final $h$ still carries what came before.

In [ ]:
def run_sequence(seq):
    h = 0.0
    for x in seq:
        h = rnn_step(x, h)
    return h

print('history [ 2,  2,  2], then x=1.0  ->  h =', round(run_sequence([ 2,  2,  2, 1.0]), 3))
print('history [-2, -2, -2], then x=1.0  ->  h =', round(run_sequence([-2, -2, -2, 1.0]), 3))
print('Same last input, different h — the state carries the past.')

### Why memory fades: the vanishing gradient, in miniature

The memory is real but **short**. For the scalar cell,

$$
\frac{\partial h_t}{\partial h_{t-1}} = \big(1 - h_t^{2}\big)\cdot W_{hh},
$$

and $\tanh'$ is at most 1 (smaller once the cell *saturates*), so one sub-1 factor is
multiplied in **per step** and the product collapses geometrically. The influence of
$x_5$ on $h_{50}$ — and the gradient flowing back the other way — effectively
**vanishes**; vanilla RNNs struggle past ~10–20 steps.

In [ ]:
# tanh' = 1 - tanh(z)^2 : peaks at 1, falls off fast.
zs = torch.linspace(-4, 4, 9)
print('z       :', '  '.join(f'{z:+.0f}' for z in zs.tolist()))
print("tanh'(z) :", '  '.join(f'{v:.2f}' for v in (1 - torch.tanh(zs) ** 2).tolist()))

# Multiply a typical sub-1 factor over many steps -> it vanishes:
print()
for steps in [5, 20, 50]:
    print(f'(0.5) ** {steps:2d}  =  {0.5 ** steps:.2e}')
print('After ~50 steps the signal from the past is all but gone.')

### From one number to a vector: the matrix RNN cell

Real RNNs use a hidden **vector** (say 128 numbers); the scalar multiplications become
`nn.Linear` layers: `W_xh = nn.Linear(n_in, n_hidden, bias=False)` plays $W_{xh}$, and
`W_hh = nn.Linear(n_hidden, n_hidden, bias=True)` plays $W_{hh}$ **and** holds $b_h$.
Implement `forward` from the spec in its comments.

In [ ]:
class RNNCell(nn.Module):
    """One vanilla-RNN step on a batch of vectors."""
    def __init__(self, n_in, n_hidden):
        super().__init__()
        self.W_xh = nn.Linear(n_in, n_hidden, bias=False)     # mixes the input
        self.W_hh = nn.Linear(n_hidden, n_hidden, bias=True)  # mixes memory (+ b_h)

    def forward(self, x_t, h_prev):
        # TODO: implement ONE vanilla-RNN step and return the new hidden state.
        #   Spec:  h_t = tanh( W_xh @ x_t + W_hh @ h_{t-1} + b_h )
        #     - x_t:    (B, n_in)        the input at this time step
        #     - h_prev: (B, n_hidden)    the previous hidden state
        #     - the bias b_h already lives inside self.W_hh
        #     - return shape: (B, n_hidden)
        raise NotImplementedError("Implement RNNCell.forward (full code in the Solutions section).")

### Loop the cell over a sequence

An RNN is this one cell applied again and again, its output fed back in as the next
`h_prev` — the parameter sharing from the lecture. Below, a fake batch of `B=4` sequences
of length `T=6`; you need to complete the TODO inside the loop.

In [ ]:
torch.manual_seed(1337)
cell = RNNCell(n_in=10, n_hidden=8)

B, T = 4, 6
xseq = torch.randn(B, T, 10)            # a batch of length-T input sequences
h = torch.zeros(B, 8)                   # h_0 = 0

for t in range(T):                      # walk through time, reusing one cell
    # TODO: one line — apply `cell` to the t-th time slice of xseq (shape (B, 10))
    #       and the current h, so the new state feeds back in next iteration.
    h = ...

if h is ...:
    print('Fill in the TODO above, then run this cell again.')
else:
    print('final hidden state shape:', tuple(h.shape))   # expect (4, 8)
    assert h.shape == (B, 8)
    print('OK — one cell, applied T times, shared weights. That is an RNN.')

## Section 3 · A language model that obeys the interface

All models we have introduced follow this form: 

```
logits, loss = model(idx, targets=None)   # idx, targets: (B, T) Long; logits: (B, T, vocab)
idx = model.generate(idx, max_new_tokens) # idx: (B, T) -> (B, T + max_new_tokens)
```

The wrapper: an **embedding** turns each character id into an `n_embd`-dim vector, the
cell is looped over the sequence, and a linear **`lm_head`** turns each $h_t$ into
next-character logits (plus cross-entropy when `targets` are given). You need to complete
`forward` using the numbered spec.

In [ ]:
class HandRolledRNN(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_hidden=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.cell = RNNCell(n_embd, n_hidden)          # the cell you wrote above
        self.lm_head = nn.Linear(n_hidden, vocab_size) # hidden -> next-char scores
        self.n_hidden = n_hidden

    def forward(self, idx, targets=None):
        # TODO: implement the forward pass from this spec (full code in Solutions).
        #   1. embed idx                -> emb of shape (B, T, n_embd)
        #   2. h_0 = zeros (B, n_hidden) on idx.device
        #   3. for t in range(T): h = self.cell(emb[:, t, :], h); collect each h
        #   4. stack the collected states -> (B, T, n_hidden); apply self.lm_head
        #      -> logits of shape (B, T, vocab)
        #   5. loss: the same flatten-and-cross-entropy move as Lab 8's bigram
        #      (None if targets is None)
        #   return logits, loss     
        raise NotImplementedError("Implement HandRolledRNN.forward (full code in the Solutions section).")

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)                              # re-read whole context
            probs = F.softmax(logits[:, -1, :], dim=-1)        # last step's scores
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

### Sanity check

An *untrained* model guesses uniformly over the vocabulary, so the loss should be close
to $-\ln(1/\text{vocab}) = \ln(\text{vocab}) \approx 4.17$ for our 65-character
alphabet.

In [ ]:
torch.manual_seed(1337)
m = HandRolledRNN(P['vocab_size']).to(device)

logits, loss = m(xb, yb)
print('logits shape:', tuple(logits.shape), '  (B, T, vocab)')
print('initial loss:', round(loss.item(), 3), ' | expected ~', round(math.log(P['vocab_size']), 3))

### It plugs into the Lab-8 loop unchanged

The model returns `(logits, loss)`, so `train_model` accepts it as-is. We only run a
short burst: the hand-rolled forward loops over time in Python — correct but slow. Next,
a version that loops in optimised C++/CUDA.

In [ ]:
torch.manual_seed(1337)
m = HandRolledRNN(P['vocab_size']).to(device)
print('Training the hand-rolled RNN for a short burst (proof it plugs in)...')
train_model(m, P['get_batch'], P['estimate_loss'],
            max_iters=300, eval_interval=150, lr=3e-3, device=device)
print('Loss dropped below the ~4.17 starting point — the loop accepted our model unchanged.')

### The practical version: `nn.GRU`

The hand-rolled RNN is slow (a Python loop) and forgetful (vanishing gradients).
PyTorch's **GRU (gated recurrent unit)** fixes both. It adds two **gates**: an **update
gate** $z_t$ (keep the old state, or overwrite it?) and a **reset gate** $r_t$ (how much
past to use in the new candidate?). The decisive line from the lecture is
$h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \tilde{h}_t$: when $z_t \to 0$ the state is
copied forward **unchanged** — an *additive* "memory highway" for gradients, the same
trick the LSTM's cell state plays.

`nn.GRU` runs the time loop in optimised code and takes the whole sequence at once:
`(B, T, n_embd)` in, `(B, T, n_hidden)` out. Two jobs below: create the GRU in
`__init__`, and finish the sampling loop in `generate`.

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_hidden=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # TODO 1: create a GRU that reads n_embd-dim inputs and keeps an
        #         n_hidden-dim state. Pass batch_first=True so it expects (B, T, feat).
        self.rnn = ...
        self.lm_head = nn.Linear(n_hidden, vocab_size)

    def forward(self, idx, targets=None):
        emb = self.token_embedding_table(idx)       # (B, T, n_embd)
        out, _ = self.rnn(emb)                      # (B, T, n_hidden) — GRU loops for us
        logits = self.lm_head(out)                  # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # The same sampling loop as the bigram's generate — write it yourself this time.
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = ...      # TODO 2: softmax over the LAST position's logits -> (B, vocab)
            idx_next = ...   # TODO 3: sample ONE id per row from probs -- the same draw as Lab 8
            idx = ...        # TODO 4: append idx_next to idx along the time dimension
        return idx

### Sanity check the GRU model

Same interface, same expected starting loss (~4.17). Compare the parameter count with the
bigram's 4,225 — that capacity is what lets it learn structure.

In [ ]:
torch.manual_seed(1337)
gru = RNNLanguageModel(P['vocab_size']).to(device)

logits, loss = gru(xb, yb)
print('GRU logits:', tuple(logits.shape), '| initial loss:', round(loss.item(), 3))
print('parameters:', sum(p.numel() for p in gru.parameters()))

## Section 4 · Train, sample, and compare

Using both models, same data, same loop we can train 3,000 steps of each, which
should take a couple of minutes on a Colab GPU.

In [ ]:
torch.manual_seed(1337)
bigram = BigramLanguageModel(P['vocab_size']).to(device)
print('Training the Lab-8 BIGRAM baseline...')
train_model(bigram, P['get_batch'], P['estimate_loss'],
            max_iters=3000, eval_interval=1000, lr=1e-2, device=device)

In [ ]:
torch.manual_seed(1337)
gru = RNNLanguageModel(P['vocab_size']).to(device)
print('Training the GRU language model...')
t0 = time.time()
train_model(gru, P['get_batch'], P['estimate_loss'],
            max_iters=3000, eval_interval=1000, lr=3e-3, device=device)
print(f'done in {time.time() - t0:.0f}s')

### Compare them — by loss, then by ear

Lower cross-entropy means higher probability on the *actual* next character. 
Sample 400 characters from each model, starting from a newline (id 0). Expect
gibberish from the bigram but real(ish) words and the line-by-line shape of a
play from the GRU.

In [ ]:
# TODO: use P['estimate_loss'] to get train/val losses for BOTH trained models,
# and print them side by side. Expect the bigram to plateau around ~2.4-2.5 and
# the GRU to land well below ~1.8.



In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)   # start token = id 0

print('=============== LAB-8 BIGRAM  (memory = 1 character) ===============')
# TODO 1: generate 400 characters from `bigram` and print them as text.
#         Lab 8's pattern: model.generate -> [0].tolist() -> P['decode'] -> print.


print()
print('=============== LAB-9 GRU  (memory = hidden state) ===============')
# TODO 2: same again for `gru`.


### What just happened

The bigram can only learn "after a `q`, a `u` is likely". The GRU's hidden state lets it
learn "I am partway through a word" or "I just opened a speaker name and should close it
with a colon" using the same data and loop, just a different model architecture.

## Recap

* You re-used the Lab-8 pipeline unchanged — the point of a fixed **interface contract**.
* You hand-rolled the recurrence $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$ and
  watched a hidden state act as memory.
* You saw the **vanishing gradient** numerically: $\tanh' \le 1$, multiplied over many
  steps, collapses to zero.
* You trained a gated **GRU**, whose additive "memory highway" eases that problem, and
  beat the bigram by roughly 0.7 nats per character.

Two stubborn limits remain: RNNs are **sequential** ($h_t$ needs $h_{t-1}$, so training
cannot parallelise across time), and they **still forget** over long spans.

### Extensions (optional)

* `max_iters` 5000 and `block_size` 64 or 128 (re-run the build cell first) — watch the
  val loss.
* Swap `nn.GRU` for `nn.LSTM`; the rest of the class runs unchanged. Does val loss move?
* Train `HandRolledRNN` for the full 3,000 steps. How much do the gates buy?

### Next time — Lab 10: attention and the transformer

On Monday we drop recurrence: **attention lets every position look directly at every
other position**, in parallel. That is the **transformer**.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — batch shapes**

Both are `(64, 32)` — `(batch_size, block_size)`. `yb` matches `xb` because there is a
target for *every* position: `yb[i, t]` is the character that follows `xb[i, t]`, so the
decoded target reads as the input shifted one right.

**Solution — the scalar recurrence**

`math.tanh` is the right tool — `x_t` and `h_prev` are plain floats; `torch.tanh` (which
a chatbot may offer) wants tensors. Check against the board: $0.762$, $-0.119$, $0.959$.

In [ ]:
def rnn_step(x_t, h_prev, W_xh=1.0, W_hh=0.5, b_h=0.0):
    # one scalar RNN step: h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
    return math.tanh(W_xh * x_t + W_hh * h_prev + b_h)

h = 0.0                                  # h_0 = 0
for x_t in [1.0, -0.5, 2.0]:
    h = rnn_step(x_t, h)
    print(f'x_t = {x_t:+.1f}   ->   h = {h:+.3f}')

**Solution — `RNNCell.forward`**

One line. The bias $b_h$ already lives inside `self.W_hh` — adding a separate `+ b`
would double-count it.

In [ ]:
class RNNCell(nn.Module):
    """One vanilla-RNN step on a batch of vectors."""
    def __init__(self, n_in, n_hidden):
        super().__init__()
        self.W_xh = nn.Linear(n_in, n_hidden, bias=False)
        self.W_hh = nn.Linear(n_hidden, n_hidden, bias=True)

    def forward(self, x_t, h_prev):
        # h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h); b_h is inside W_hh
        h_t = torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))
        return h_t

print('RNNCell solution loaded.')

**Solution — looping the cell through time**

`xseq[:, t, :]` is the $t$-th time slice, `(B, 10)`; assigning back to `h` feeds the
state forward.

In [ ]:
torch.manual_seed(1337)
cell = RNNCell(n_in=10, n_hidden=8)

B, T = 4, 6
xseq = torch.randn(B, T, 10)
h = torch.zeros(B, 8)

for t in range(T):
    h = cell(xseq[:, t, :], h)          # the new state feeds back in

print('final hidden state shape:', tuple(h.shape))   # (4, 8)
assert h.shape == (B, 8)

**Solution — `HandRolledRNN.forward`**

The loop collects a hidden state at *every* position because the loss needs a prediction
at every position. `torch.stack(outs, dim=1)` turns the list of `(B, n_hidden)` states
into one `(B, T, n_hidden)` tensor.

In [ ]:
class HandRolledRNN(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_hidden=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.cell = RNNCell(n_embd, n_hidden)
        self.lm_head = nn.Linear(n_hidden, vocab_size)
        self.n_hidden = n_hidden

    def forward(self, idx, targets=None):
        B, T = idx.shape
        emb = self.token_embedding_table(idx)                  # (B, T, n_embd)
        h = torch.zeros(B, self.n_hidden, device=idx.device)   # h_0 = 0
        outs = []
        for t in range(T):
            h = self.cell(emb[:, t, :], h)                     # (B, n_hidden)
            outs.append(h)
        hidden = torch.stack(outs, dim=1)                      # (B, T, n_hidden)
        logits = self.lm_head(hidden)                          # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print('HandRolledRNN solution loaded.')

**Solution — the GRU model**

`nn.GRU` returns `(output, h_n)`, hence `out, _`. And `torch.multinomial` *samples*; the
`argmax` a chatbot might suggest takes the single likeliest character every time, which
collapses into repetitive loops — sampling keeps the variety.

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_hidden=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.rnn = nn.GRU(n_embd, n_hidden, batch_first=True)
        self.lm_head = nn.Linear(n_hidden, vocab_size)

    def forward(self, idx, targets=None):
        emb = self.token_embedding_table(idx)
        out, _ = self.rnn(emb)
        logits = self.lm_head(out)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print('RNNLanguageModel (GRU solution) defined.')

**Solution — bigram vs GRU losses**

The gap is roughly 0.7 nats per character; cross-entropy is a log scale, so the GRU puts
about $e^{0.7} \approx 2$ times the probability on the actual next character.

In [ ]:
bigram_losses = P['estimate_loss'](bigram)
gru_losses    = P['estimate_loss'](gru)

print(f"bigram | train {bigram_losses['train']:.4f} | val {bigram_losses['val']:.4f}")
print(f"GRU    | train {gru_losses['train']:.4f} | val {gru_losses['val']:.4f}")

**Solution — sample and decode**

`generate` returns `(1, 401)` ids; `[0].tolist()` unwraps the batch dimension into the
plain list `P['decode']` expects.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)   # start token = id 0

print('=============== LAB-8 BIGRAM  (memory = 1 character) ===============')
print(P['decode'](bigram.generate(context, 400)[0].tolist()))

print()
print('=============== LAB-9 GRU  (memory = hidden state) ===============')
print(P['decode'](gru.generate(context, 400)[0].tolist()))